# RAG PDF Assistant - Option 1: Ollama (fully local)
- Upload one or more PDFs
- Ingestion: load, split, embed, store in Chroma
- Retrieval + augmentation + generation with LangChain

**Run this notebook locally (Jupyter / VS Code), not on Colab** - Ollama runs on your machine.

Before running, in your terminal:
```
ollama pull nomic-embed-text
ollama list   # should show llama3.2:3b and nomic-embed-text
```
Make sure Ollama is running (the menu-bar app, or `ollama serve`).

In [1]:
!pip install -q langchain langchain-core langchain-community langchain-text-splitters langchain-ollama chromadb pypdf gradio

In [2]:
# import ollama
# ollama.pull("nomic-embed-text")

In [3]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Community integrations
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

# Text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

import gradio as gr

/var/folders/2p/cbc008f94fn08q7sb7wq_k740000gn/T/ipykernel_3618/1209318436.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
VECTOR_DB = None

def load_pdfs(files):
    documents = []
    for f in files:
        path = f if isinstance(f, str) else f.name   # works across Gradio versions
        loader = PyPDFLoader(path)
        documents.extend(loader.load())
    return documents

def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,      # max characters per chunk
        chunk_overlap=150,   # characters shared with the previous chunk
    )
    return splitter.split_documents(documents)

def create_vectorstore(chunks):
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    # In-memory Chroma: every ingestion starts fresh, so no stale data or
    # embedding-dimension clashes from earlier runs.
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
    )
    return vectorstore

In [5]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question using only the context below.
If the answer is not present in the context, then say:
"I could not find the answer in the provided document(s)"

Context:
{context}

Question:
{question}

Answer very clearly and concisely
""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def build_rag_chain(vectorstore):
    # Retriever is only initialised here; it runs when the chain is invoked.
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    llm = ChatOllama(model="llama3.2:3b", temperature=0)
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | RAG_PROMPT
        | llm
    )
    return rag_chain

def ingest_pdf(files):
    global VECTOR_DB
    if not files:
        return "Upload files first"
    documents = load_pdfs(files)
    chunks = split_documents(documents)
    VECTOR_DB = create_vectorstore(chunks)
    return f"Successfully ingested {len(chunks)} chunks from {len(files)} PDF(s)"

def ask_question(question):
    if VECTOR_DB is None:
        yield "Upload the files and click Ingest first"
        return
    rag_chain = build_rag_chain(VECTOR_DB)
    response = rag_chain.invoke(question)   # the pipeline actually executes here

    # Gradio streaming effect
    text = response.content
    partial = ""
    for char in text:
        partial += char
        yield partial

In [ ]:
with gr.Blocks(title="LangChain RAG PDF Assistant (Ollama)") as demo:
    gr.Markdown("# LangChain RAG PDF Assistant (Ollama)")
    gr.Markdown("Upload PDF documents and ask questions grounded strictly in their content")
    with gr.Row():
        pdf_files = gr.File(file_types=[".pdf"], file_count="multiple", label="Upload PDF files")
    ingest_btn = gr.Button("Ingest PDF")
    ingest_status = gr.Textbox(label="Ingestion Status")
    ingest_btn.click(ingest_pdf, inputs=[pdf_files], outputs=[ingest_status])

    gr.Markdown("----")

    question = gr.Textbox(label="Ask any question", placeholder="What does the document talk about?")
    ask_btn = gr.Button("Ask")
    answer_box = gr.Markdown(label="Answer")
    ask_btn.click(ask_question, inputs=[question], outputs=[answer_box])

demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
